# D4-LensPINN v4 · Physics-Informed Gravitational Lens Substructure
*Author: [AUTHOR], [INSTITUTION]*

---
## VanillaLensPINN Control Experiment
This notebook trains and evaluates **VanillaLensPINN**: a non-equivariant version of the D4-PINN model. 
It serves as a causal control for the mechanistic interpretability experiment.


In [ ]:
import subprocess, sys

# ── numpy 1.26.4 guard + escnn install ───────────────────────────────────────
try:
    import numpy as np
    numpy_ok = np.__version__.startswith("1.26")
except Exception:
    numpy_ok = False

if not numpy_ok:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "numpy==1.26.4", "escnn", "gdown"], check=True)
    print("✅ Installed — restarting kernel now...")
    import IPython
    IPython.get_ipython().kernel.do_shutdown(restart=True)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "escnn", "gdown"], check=True)
    print(f"✅ numpy {np.__version__} already correct — no restart needed")

import numpy as np
assert np.__version__.startswith("1.26"), \
    f"Wrong numpy: {np.__version__} — re-run Cell 0"
print(f"✅ numpy {np.__version__}")

# ── escnn imports ─────────────────────────────────────────────────────────────
from escnn import gspaces
import escnn.nn as enn
from escnn.nn import GeometricTensor

# ── All other MI imports ──────────────────────────────────────────────────────
import os, sys, json, copy, gc, timeit, warnings
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import torch
import torch.nn.functional as F
from torch import nn
from sklearn.metrics import roc_auc_score
from scipy import stats
warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True

# ── Device config ─────────────────────────────────────────────────────────────
DEVICE_PINN   = 'cuda:0'
DEVICE_RESNET = 'cuda:1' if torch.cuda.device_count() > 1 else 'cuda:0'
print(f"GPU count: {torch.cuda.device_count()}")
print(f"PINN device: {DEVICE_PINN} | ResNet device: {DEVICE_RESNET}")

# ── Paths ─────────────────────────────────────────────────────────────────────
CKPT_DIR    = '/kaggle/input/datasets/[AUTHOR]/d4-pinn-and-resnet'
PINN_CKPT   = os.path.join(CKPT_DIR, 'd4_phase2_best.pth')
RESNET_CKPT = os.path.join(CKPT_DIR, 'resnet18_baseline_best.pth')
OUT_DIR     = '/kaggle/working/mi_experiment'
os.makedirs(OUT_DIR, exist_ok=True)

N_SUBSET = 200  # 67+67+66 per class
print("✅ Cell 0 complete")

## Cell 2 — MASTER TRUTH TABLE — Claims Permitted in Paper
| Model | AUC | Outcome | Mechanism |
|---|---|---|---|
| D4LensPINN | 0.9786 | ROUTING | Equivariance circuit in EfficientNetV2 |
| VanillaLensPINN | 0.9776 | INDETERMINATE (chain) | GAP-ratio artifact; head verdict requires per-layer fresh-g·x runs |
| ResNet18 | 0.9182 | COLLAPSE | Global Average Pooling invariant by fiat |

**AUC note:** VanillaLensPINN (0.9776) and D4LensPINN (0.9786) differ by 0.001 — within noise. 
The paper's mechanistic contribution stands independently of AUC.

**Thresholds:** report results at three thresholds (0.5/0.2, 0.6/0.3, 0.4/0.15).


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────
import os, glob, json, copy, datetime
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.nn.utils import clip_grad_norm_
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms, datasets
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights, resnet18
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, roc_curve, auc, confusion_matrix
import matplotlib.pyplot as plt
try:
    from escnn import nn as enn, gspaces
    from escnn.nn import GeometricTensor
except ImportError:
    print("Warning: escnn not found. Ensure cell 1 ran successfully.")

# ── Global hyperparameters ─────────────────────────────────────────────────
SEED           = 42
BATCH          = 32
N_SUBSET       = 200
IMG_SIZE       = 150
NUM_CLASSES    = 3
OUT_DIR        = '/kaggle/working'
CKPT_DIR       = '/kaggle/working'
VANILLA_CKPT   = '/kaggle/working/vanilla_phase2_best.pth'
DEVICE_VANILLA = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# ── D4 group transforms (Bug C: .contiguous() required) ──────────────────
GROUP_ELEMS = ['e', 'r90', 'r180', 'r270', 'flip_h', 'flip_h_r90', 'flip_h_r180', 'flip_h_r270']

D4_TRANSFORMS = {
    'e':           lambda x: x.contiguous(),
    'r90':         lambda x: torch.rot90(x, k=1, dims=[2, 3]).contiguous(),
    'r180':        lambda x: torch.rot90(x, k=2, dims=[2, 3]).contiguous(),
    'r270':        lambda x: torch.rot90(x, k=3, dims=[2, 3]).contiguous(),
    'flip_h':      lambda x: torch.flip(x, dims=[3]).contiguous(),
    'flip_h_r90':  lambda x: torch.rot90(torch.flip(x, dims=[3]), k=1, dims=[2, 3]).contiguous(),
    'flip_h_r180': lambda x: torch.rot90(torch.flip(x, dims=[3]), k=2, dims=[2, 3]).contiguous(),
    'flip_h_r270': lambda x: torch.rot90(torch.flip(x, dims=[3]), k=3, dims=[2, 3]).contiguous(),
}

torch.manual_seed(SEED)
np.random.seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Globals initialized. Device: {DEVICE_VANILLA}")


In [ ]:
# ── Dataset download (if not already extracted) ─────────────────────────────
import os, zipfile

ZIP_PATH = "/kaggle/working/dataset.zip"
EXTRACT_MARKER = "/kaggle/working/.extracted"

def zip_is_valid(path):
    try:
        with zipfile.ZipFile(path, "r") as z:
            return z.testzip() is None
    except Exception:
        return False

if not os.path.exists(EXTRACT_MARKER):
    if not os.path.exists(ZIP_PATH) or not zip_is_valid(ZIP_PATH):
        print("Downloading dataset...")
        ret = os.system(
            "gdown https://drive.google.com/uc?id=1ZEyNMEO43u3qhJAwJeBZxFBEYc_pVYZQ "
            "-O /kaggle/working/dataset.zip"
        )
        if ret != 0 or not zip_is_valid(ZIP_PATH):
            raise RuntimeError("Download failed or corrupt. Check the Drive ID.")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall("/kaggle/working/")
    open(EXTRACT_MARKER, "w").close()
    print("Extraction complete.")
else:
    print("Dataset already extracted — skipping.")


In [ ]:
# ── Physics Engine ───────────────────────────────────────────────────
class PhysicsPreprocess(nn.Module):
    def __init__(self):
        super().__init__()
        kx = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3)
        ky = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).view(1, 1, 3, 3)
        self.register_buffer('kx', kx)
        self.register_buffer('ky', ky)

    def forward(self, x):
        dx = F.conv2d(x, self.kx, padding=1)
        dy = F.conv2d(x, self.ky, padding=1)
        return torch.cat([dx, dy], dim=1)

def _next_pow2(n):
    return 1 << (n - 1).bit_length()

class PoissonSolverFFT(nn.Module):
    def __init__(self, H=150, W=150):
        super().__init__()
        H_padded, W_padded = _next_pow2(H), _next_pow2(W)
        self.H_orig, self.W_orig = H, W
        self.H_padded, self.W_padded = H_padded, W_padded

        ky = torch.fft.fftfreq(H_padded).reshape(-1, 1)
        kx = torch.fft.fftfreq(W_padded).reshape(1, -1)
        k2 = kx**2 + ky**2
        k2[0, 0] = 1.0  
        inv_k2 = 1.0 / (4 * (np.pi**2) * k2)
        inv_k2[0, 0] = 0.0
        self.register_buffer('inv_k2', inv_k2)

    def forward(self, kappa):
        kappa_pad = F.pad(kappa, (0, self.W_padded - self.W_orig, 0, self.H_padded - self.H_orig))
        kappa_ft = torch.fft.fft2(kappa_pad)
        psi_ft = kappa_ft * self.inv_k2
        psi_pad = torch.fft.ifft2(psi_ft).real
        return psi_pad[:, :, :self.H_orig, :self.W_orig]

class DeflectionField(nn.Module):
    def forward(self, psi):
        alpha_y = torch.gradient(psi, dim=2)[0]
        alpha_x = torch.gradient(psi, dim=3)[0]
        return torch.cat([alpha_x, alpha_y], dim=1)

class InverseLensLayer(nn.Module):
    def __init__(self, H=150, W=150):
        super().__init__()
        y, x = torch.meshgrid(torch.linspace(-1, 1, H), torch.linspace(-1, 1, W), indexing='ij')
        self.register_buffer('grid', torch.stack([x, y], dim=-1).unsqueeze(0))

    def forward(self, I, alpha):
        alpha_trans = alpha.permute(0, 2, 3, 1) # B,H,W,2
        sampling_grid = self.grid - alpha_trans
        S_hat = F.grid_sample(I, sampling_grid, align_corners=True)
        return S_hat, I - S_hat

# ── Neural Components ────────────────────────────────────────────────
class EfficientNetV2Head(nn.Module):
    def __init__(self, num_classes=3, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Conv2d(4, 3, kernel_size=1, bias=False),
            nn.BatchNorm2d(3), nn.SiLU()
        )
        self.register_buffer('imgnet_mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('imgnet_std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        
        base = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.IMAGENET1K_V1)
        for p in base.parameters(): p.requires_grad = False
        for name, p in base.named_parameters():
            if any(tag in name for tag in ['features.3','features.4','features.5','features.6','features.7','classifier']):
                p.requires_grad = True
                
        self.features = base.features
        self.avgpool = base.avgpool
        in_features = base.classifier[-1].in_features
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_features, num_classes))

    def forward(self, x):
        x = self.input_proj(x)
        x = (x - self.imgnet_mean) / self.imgnet_std
        x = self.features(x)
        x = self.avgpool(x).flatten(1)
        return self.head(x)

class _VanillaResBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(cin, cout, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout)
        )
        self.proj = nn.Conv2d(cin, cout, 1, stride=stride, bias=False) if stride != 1 or cin != cout else None
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.body(x) + (self.proj(x) if self.proj else x))

class VanillaUNet(nn.Module):
    def __init__(self, H=150, W=150):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(2, 64, 7, stride=2, padding=3, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.enc2 = _VanillaResBlock(64, 128, stride=2)
        self.enc3 = _VanillaResBlock(128, 256, stride=2)
        self.bot  = _VanillaResBlock(256, 384, stride=1)
        self.dec3 = nn.Sequential(nn.Conv2d(384+256, 256, 3, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(inplace=True))
        self.dec2 = nn.Sequential(nn.Conv2d(256+128, 128, 3, padding=1, bias=False), nn.BatchNorm2d(128), nn.ReLU(inplace=True))
        self.dec1 = nn.Sequential(nn.Conv2d(128+64, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.kappa_out = nn.Sequential(nn.Conv2d(64, 1, 1), nn.Softplus())

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        b = self.bot(e3)
        d = self.dec3(torch.cat([b, e3], dim=1))
        d = self.dec2(torch.cat([F.interpolate(d, size=e2.shape[-2:], mode='bilinear'), e2], dim=1))
        d = self.dec1(torch.cat([F.interpolate(d, size=e1.shape[-2:], mode='bilinear'), e1], dim=1))
        d = F.interpolate(d, size=(150, 150), mode='bilinear')
        return self.kappa_out(d)

class VanillaLensPINN(nn.Module):
    def __init__(self, H=150, W=150, num_classes=3):
        super().__init__()
        self.preprocess = PhysicsPreprocess()
        self.d4unet     = VanillaUNet(H, W)
        self.poisson    = PoissonSolverFFT(H, W)
        self.deflection = DeflectionField()
        self.inv_lens   = InverseLensLayer(H, W)
        self.classifier = EfficientNetV2Head(num_classes)
        self.handoff_probe = nn.Identity()

    def forward(self, I):
        X_in = self.preprocess(I)
        kappa = self.d4unet(X_in)
        psi = self.poisson(kappa)
        alpha = self.deflection(psi)
        S_hat, R = self.inv_lens(I, alpha)
        X_cls = self.handoff_probe(torch.cat([I, kappa, S_hat, R], dim=1))
        logits = self.classifier(X_cls)
        return logits, kappa

# ── Prediction Helpers ───────────────────────────────────────────────
@torch.no_grad()
def predict_no_tta(model, loader, device, is_pinn=True):
    model.eval()
    all_probs, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        out = model(imgs)
        logits = out[0] if isinstance(out, tuple) else out
        all_probs.append(torch.softmax(logits, -1).cpu())
        all_labels.append(labels)
    return torch.cat(all_probs), torch.cat(all_labels)

@torch.no_grad()
def predict_with_tta(model, loader, device, is_pinn=True):
    model.eval()
    all_probs, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        batch_probs = []
        for k in range(4):
            for flip in [False, True]:
                x = torch.rot90(imgs, k=k, dims=(-2, -1))
                if flip: x = torch.flip(x, dims=[-1])
                out = model(x)
                logits = out[0] if isinstance(out, tuple) else out
                batch_probs.append(torch.softmax(logits, -1).cpu())
        all_probs.append(torch.stack(batch_probs).mean(0))
        all_labels.append(labels)
    return torch.cat(all_probs), torch.cat(all_labels)


In [ ]:
vanilla_model = VanillaLensPINN(H=IMG_SIZE, W=IMG_SIZE, num_classes=NUM_CLASSES).to(DEVICE_VANILLA)
print(f"VanillaLensPINN params: {sum(p.numel() for p in vanilla_model.parameters()):,}")
assert isinstance(vanilla_model.d4unet, VanillaUNet)
print("Structural verification Passed.")


In [ ]:
# Cell 5 - Data loading and canonical split (same path as d4lenspinn_mi_working)
from pathlib import Path
from torchvision import datasets, transforms
from torch.utils.data import Dataset, Subset, DataLoader
from sklearn.model_selection import StratifiedShuffleSplit

def _to_1x150x150_tensor(arr):
    arr = np.asarray(arr)

    if arr.ndim == 2:
        t = torch.from_numpy(arr).float().unsqueeze(0)
    elif arr.ndim == 3:
        if arr.shape[0] in (1, 3):
            t = torch.from_numpy(arr).float()
        elif arr.shape[-1] in (1, 3):
            t = torch.from_numpy(np.transpose(arr, (2, 0, 1))).float()
        else:
            raise ValueError(f"Unsupported 3D sample shape {arr.shape}")
    else:
        raise ValueError(f"Unsupported sample ndim={arr.ndim}, shape={arr.shape}")

    if t.shape[0] != 1:
        t = t.mean(dim=0, keepdim=True)

    if tuple(t.shape[-2:]) != (IMG_SIZE, IMG_SIZE):
        t = F.interpolate(
            t.unsqueeze(0), size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False
        ).squeeze(0)

    return t

class NpyExpandedFolderDataset(Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.class_names = sorted([d.name for d in Path(root_dir).iterdir() if d.is_dir()])
        self.class_to_idx = {c: i for i, c in enumerate(self.class_names)}
        self.records = []

        for cls in self.class_names:
            cls_idx = self.class_to_idx[cls]
            cls_dir = Path(root_dir) / cls
            npy_files = sorted(cls_dir.rglob('*.npy'))
            for fp in npy_files:
                arr = np.load(fp, mmap_mode='r')
                shape = arr.shape

                if arr.ndim == 2:
                    self.records.append((str(fp), cls_idx, 'single', 0))
                elif arr.ndim == 3:
                    if shape[0] in (1, 3) or shape[-1] in (1, 3):
                        self.records.append((str(fp), cls_idx, 'single', 0))
                    else:
                        for i in range(shape[0]):
                            self.records.append((str(fp), cls_idx, 'n_h_w', i))
                elif arr.ndim == 4:
                    if shape[1] in (1, 3):
                        for i in range(shape[0]):
                            self.records.append((str(fp), cls_idx, 'n_c_h_w', i))
                    elif shape[-1] in (1, 3):
                        for i in range(shape[0]):
                            self.records.append((str(fp), cls_idx, 'n_h_w_c', i))
                    else:
                        raise ValueError(f"Unsupported 4D npy shape {shape} in {fp}")
                else:
                    raise ValueError(f"Unsupported npy ndim={arr.ndim} shape={shape} in {fp}")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        path, label, mode, i = self.records[idx]
        arr = np.load(path)

        if mode == 'single':
            sample = arr
        elif mode in ('n_h_w', 'n_c_h_w', 'n_h_w_c'):
            sample = arr[i]
        else:
            raise RuntimeError(f"Unknown mode: {mode}")

        x = _to_1x150x150_tensor(sample)
        return x, label

REBUILD_FROM_TRAIN_SPLIT = True
TRAIN_ROOT = '/kaggle/working/dataset/train'

if REBUILD_FROM_TRAIN_SPLIT:
    assert os.path.isdir(TRAIN_ROOT), f"Missing train root: {TRAIN_ROOT}"

    _tfm = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
    ])

    try:
        _full_ds = datasets.ImageFolder(TRAIN_ROOT, transform=_tfm)
        _build_mode = 'ImageFolder'
        _labels_full = np.array([y for _, y in _full_ds.samples], dtype=np.int64)
    except FileNotFoundError:
        _full_ds = NpyExpandedFolderDataset(TRAIN_ROOT)
        _build_mode = 'NpyExpandedFolderDataset'
        _labels_full = np.array([_full_ds.records[i][1] for i in range(len(_full_ds))], dtype=np.int64)

    _idx_all = np.arange(len(_full_ds))
    _sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    _train_idx, _temp_idx = next(_sss1.split(_idx_all, _labels_full))

    _temp_labels = _labels_full[_temp_idx]
    _sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED)
    _val_rel, _test_rel = next(_sss2.split(np.zeros(len(_temp_idx)), _temp_labels))

    _val_idx = _temp_idx[_val_rel]
    _test_idx = _temp_idx[_test_rel]

    train_dataset = Subset(_full_ds, _train_idx)
    val_dataset = Subset(_full_ds, _val_idx)
    test_dataset = Subset(_full_ds, _test_idx)

    class_names = list(_full_ds.classes) if hasattr(_full_ds, 'classes') else sorted(list(_full_ds.class_to_idx.keys()))
    _test_labels = _labels_full[_test_idx]
    _vals, _cnts = np.unique(_test_labels, return_counts=True)
    _dist = {int(k): int(v) for k, v in zip(_vals, _cnts)}

    print(f"Rebuilt split from: {TRAIN_ROOT} ({_build_mode})")
    print(f"Split sizes (train/val/test): {len(train_dataset)} / {len(val_dataset)} / {len(test_dataset)}")
    print(f"Test class distribution (index:count): {_dist}")
    print(f"Class mapping: {class_names}")
else:
    raise RuntimeError("This notebook expects REBUILD_FROM_TRAIN_SPLIT=True for parity.")

train_loader = DataLoader(
    train_dataset, batch_size=BATCH, shuffle=True,
    num_workers=2, pin_memory=True, persistent_workers=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH, shuffle=False,
    num_workers=2, pin_memory=True, persistent_workers=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH, shuffle=False,
    num_workers=2, pin_memory=True, persistent_workers=True
)

_x0, _y0 = test_dataset[0]
assert isinstance(_x0, torch.Tensor), f"Expected torch.Tensor, got {type(_x0)}"
assert _x0.shape == (1, IMG_SIZE, IMG_SIZE), f"Expected (1,{IMG_SIZE},{IMG_SIZE}), got {_x0.shape}"
assert _x0.dtype == torch.float32, f"Expected float32, got {_x0.dtype}"
print(f"Evaluation dataset: {len(test_dataset)} images, shape: {_x0.shape}, label: {_y0}")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 12 — Loss, Optimizer, Train Loop  (FIXED)
# Changes vs current:
#   1. FocalLoss replaced by nn.CrossEntropyLoss ✅ (already done)
#   2. wd added as param to get_optimizer_and_scheduler ✅ (NEW FIX)
#   3. wd + lambda_poisson added as params to train_model ✅ (NEW FIX)
#   4. Warmup + CosineAnnealing replaces OneCycleLR ✅ (matches Task 1)
# ════════════════════════════════════════════════════════════════════

class PhysicsLoss(nn.Module):
    def __init__(self, H=150, W=150,
                 lambda_tv=0.005, lambda_l1=0.001,
                 lambda_ctr=0.002, lambda_poisson=0.01):  # ← lambda_poisson tunable
        super().__init__()
        self.lambda_tv      = lambda_tv
        self.lambda_l1      = lambda_l1
        self.lambda_ctr     = lambda_ctr
        self.lambda_poisson = lambda_poisson
        yy = torch.linspace(-1, 1, H)
        xx = torch.linspace(-1, 1, W)
        GY, GX = torch.meshgrid(yy, xx, indexing='ij')
        self.register_buffer('r2', (GX**2 + GY**2).unsqueeze(0).unsqueeze(0))
        lap = torch.zeros(1, 1, 3, 3)
        lap[0, 0, 1, 1] = -4.0
        lap[0, 0, 0, 1] = lap[0, 0, 2, 1] = 1.0
        lap[0, 0, 1, 0] = lap[0, 0, 1, 2] = 1.0
        self.register_buffer('lap_kernel', lap)

    def forward(self, kappa, psi):
        tv_x = (kappa[..., 1:]    - kappa[..., :-1]).abs().mean()
        tv_y = (kappa[..., 1:, :] - kappa[..., :-1, :]).abs().mean()
        ctr  = (kappa * self.r2).mean()
        lap_psi   = F.conv2d(psi, self.lap_kernel, padding=1)
        poisson_r = (lap_psi - 2.0 * kappa).pow(2).mean()
        return (self.lambda_tv * (tv_x + tv_y)
                + self.lambda_l1 * kappa.mean()
                + self.lambda_ctr * ctr
                + self.lambda_poisson * poisson_r)


def mixup_batch(imgs, labels, alpha):
    if alpha <= 0: return imgs, labels, labels, 1.0
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(imgs.size(0), device=imgs.device)
    return lam*imgs + (1-lam)*imgs[idx], labels, labels[idx], lam


def get_optimizer_and_scheduler(model, epochs, warmup_epochs=3, lr=1e-3, wd=1e-4):
    # ↑ wd is now a parameter — Optuna's BEST_WD flows through here
    params = [p for p in model.parameters() if p.requires_grad]
    opt    = AdamW(params, lr=lr, weight_decay=wd)           # ← wd wired in
    warmup = LinearLR(opt, start_factor=1/max(warmup_epochs,1),
                      end_factor=1.0, total_iters=max(warmup_epochs,1))
    cosine = CosineAnnealingLR(opt, T_max=max(epochs-warmup_epochs,1), eta_min=1e-7)
    sched  = SequentialLR(opt, schedulers=[warmup, cosine],
                          milestones=[warmup_epochs])
    return opt, sched


@torch.no_grad()
def evaluate(model, loader, criterion, phys, device, is_pinn):
    model.eval()
    tot_loss, correct, n = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        if is_pinn:
            X_in  = model.preprocess(imgs)
            kappa = model.d4unet(X_in)
            psi   = model.poisson(kappa)
            alpha = model.deflection(psi)
            S_hat, R = model.inv_lens(imgs, alpha)
            X_cls  = torch.cat([imgs, kappa, S_hat, R], dim=1)
            with torch.amp.autocast('cuda'):
                logits = model.classifier(X_cls)
            loss = criterion(logits, labels) + phys(kappa, psi)
        else:
            with torch.amp.autocast('cuda'):
                logits = model(imgs)
            loss = criterion(logits, labels)
        tot_loss += loss.item() * imgs.size(0)
        correct  += (logits.argmax(1) == labels).sum().item()
        n        += imgs.size(0)
    return tot_loss / n, correct / n



def train_model(model, train_loader, val_loader, device, epochs,
                is_pinn=True, use_phys_loss=True,
                label='model', mixup_alpha=0.0, label_smooth=0.0,
                lr=1e-3, wd=1e-4,                   # ← wd param (NEW)
                lambda_poisson=0.01,                 # ← lambda_poisson param (NEW)
                warmup_epochs=3,
                start_epoch=0, ckpt_path=None):      # ← Resume params (NEW)

    criterion = nn.CrossEntropyLoss()                # plain CE — matches Task 1
    phys      = PhysicsLoss(H=IMG_SIZE, W=IMG_SIZE,
                            lambda_poisson=lambda_poisson).to(device)  # ← tunable
                            
    # Load model weights if resuming
    if start_epoch > 0 and ckpt_path and os.path.exists(ckpt_path):
        print(f"Resuming {label} from {ckpt_path} at epoch {start_epoch}...")
        model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))

    opt, sched = get_optimizer_and_scheduler(
                    model, epochs,
                    warmup_epochs=warmup_epochs,
                    lr=lr, wd=wd)                    # ← wd flows through
                    
    # Fast-forward scheduler if resuming
    for _ in range(start_epoch):
        sched.step()

    scaler   = torch.amp.GradScaler('cuda')
    hist     = {k: [] for k in ['train_loss','val_loss','train_acc','val_acc']}
    best_val = float('inf')

    # If we are completely done, don't run loop
    if start_epoch >= epochs:
        print(f"[{label}] Already trained up to {start_epoch}/{epochs} epochs. Skipping training loop.")

    for ep in range(start_epoch + 1, epochs + 1):
        model.train()
        ep_loss, ep_corr, ep_n = 0.0, 0, 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            imgs_m, la, lb, lam = mixup_batch(imgs, labels, mixup_alpha)
            opt.zero_grad()

            if is_pinn:
                X_in    = model.preprocess(imgs_m)
                kappa   = model.d4unet(X_in)
                psi     = model.poisson(kappa)
                alpha_d = model.deflection(psi)
                S_hat, R = model.inv_lens(imgs_m, alpha_d)
                X_cls   = torch.cat([imgs_m, kappa, S_hat, R], dim=1)
                phys_v  = phys(kappa, psi) if use_phys_loss else \
                          torch.tensor(0.0, device=device)
                with torch.amp.autocast('cuda'):
                    logits   = model.classifier(X_cls)
                    cls_loss = lam*criterion(logits,la) + (1-lam)*criterion(logits,lb)
                loss = cls_loss.float() + phys_v
            else:
                with torch.amp.autocast('cuda'):
                    logits = model(imgs_m)
                    loss   = lam*criterion(logits,la) + (1-lam)*criterion(logits,lb)

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            

            dominant = la if lam >= 0.5 else lb
            ep_loss += loss.item() * imgs.size(0)
            ep_corr += (logits.argmax(1) == dominant).sum().item()
            ep_n    += imgs.size(0)

        tr_loss, tr_acc = ep_loss/ep_n, ep_corr/ep_n
        sched.step()
        vl_loss, vl_acc = evaluate(model, val_loader, criterion, phys, device, is_pinn)

        for k, v in zip(['train_loss','val_loss','train_acc','val_acc'],
                        [tr_loss, vl_loss, tr_acc, vl_acc]):
            hist[k].append(v)

        if vl_loss < best_val:
            best_val = vl_loss
            # Always save as _best.pth so it can be resumed incrementally or used for tracking
            torch.save(model.state_dict(), f'{label}_best.pth')

        if ep % 5 == 0 or ep == 1:
            print(f'[{label}] E{ep:02d}/{epochs} '
                  f'tr={tr_loss:.4f}/{tr_acc:.3f} '
                  f'vl={vl_loss:.4f}/{vl_acc:.3f} '
                  f'lr={opt.param_groups[0]["lr"]:.2e}')

    if os.path.exists(f'{label}_best.pth'):
        model.load_state_dict(torch.load(f'{label}_best.pth',
                              map_location=device, weights_only=True))
    print(f'[{label}] Done.')
    return hist


print("=== Phase 1: Training VanillaLensPINN (Classifier Only) ===")

import os

# --- Resuming Logic ---
# To resume training dynamically from a specific epoch, adjust start_epoch_p1.
start_epoch_p1 = 15
ckpt_path_p1 = '/kaggle/input/datasets/[NAME]26189/vanilla-weights/vanilla_phase2_best (1).pth'

if os.path.exists(ckpt_path_p1):
    print(f"Found phase 1 checkpoint at {ckpt_path_p1}. Checking epoch config...")
    # NOTE: Set start_epoch_p1 to your interrupted epoch before running this cell if resuming.
    # Example: start_epoch_p1 = 10

hist_p1 = train_model(
    vanilla_model, train_loader, val_loader,
    device=DEVICE_VANILLA, epochs=15,
    is_pinn=True, use_phys_loss=False,
    label='vanilla_phase1',
    lr=7e-4, wd=1e-3,
    start_epoch=start_epoch_p1,
    ckpt_path=ckpt_path_p1
)


In [ ]:
import os

print("=== Phase 2: Full VanillaLensPINN Training (with Physics) ===")

# --- Resuming Logic ---
# To resume training dynamically from a specific epoch, adjust start_epoch.
# If the kernel disconnected at epoch 15, set start_epoch = 15. The loop will run 16 -> 40.
start_epoch = 60
ckpt_path = '/kaggle/input/datasets/[NAME]26189/vanilla-weights/vanilla_phase2_best (1).pth'

if os.path.exists(ckpt_path):
    print(f"Found phase 2 checkpoint at {ckpt_path}. Checking epoch config...")
    # NOTE: Set start_epoch to your interrupted epoch before running this cell if resuming.
    # Example: start_epoch = 25

hist_p2 = train_model(
    vanilla_model, train_loader, val_loader,
    device=DEVICE_VANILLA, epochs=60,
    is_pinn=True, use_phys_loss=True,
    label='vanilla_phase2',
    lr=4e-4, wd=1e-3,
    start_epoch=start_epoch,
    ckpt_path=ckpt_path
)


In [ ]:
import zipfile
torch.save(vanilla_model.state_dict(), 'vanilla_phase2_best.pth')
with zipfile.ZipFile('vanilla_model_weights.zip', 'w') as zf:
    zf.write('vanilla_phase2_best.pth')
print("Model weights zipped: vanilla_model_weights.zip")


In [ ]:
# Run this in the VanillaLensPINN notebook after loading vanilla_model
# Compare H09 probe accuracy: D4 vs Vanilla
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
import numpy as np
N_PCA = 100  # Added dynamically to resolve NameError
VANILLA_PROBE_HOOKS = {
    'H09_poisson': vanilla_model.poisson,
    'H12_HANDOFF': vanilla_model.handoff_probe,
}

_van_feats  = {h: [] for h in VANILLA_PROBE_HOOKS}
_van_labels = []

# Load or sample MI subset
labels_all = np.array([test_dataset[i][1] for i in range(len(test_dataset))])
subset_indices = []
counts = {0: 67, 1: 67, 2: 66}
torch.manual_seed(SEED); np.random.seed(SEED)
for cls, n in counts.items():
    cls_idx = np.where(labels_all == cls)[0]
    subset_indices.extend(np.random.choice(cls_idx, size=n, replace=False).tolist())
subset_indices = sorted(subset_indices)

def cache_activations(model, x, hook_specs, device):
    """Single forward pass. Returns (logits, cache_dict)."""
    cache = {}
    hooks = []
    from escnn.nn import GeometricTensor

    for name, module in hook_specs.items():
        def _make_hook(n):
            def _hook(mod, inp, output):
                if isinstance(output, tuple):
                    cache[n] = tuple(
                        t.detach().clone() if isinstance(t, torch.Tensor) else t
                        for t in output
                    )
                    cache[n + '__is_tuple'] = True
                elif isinstance(output, GeometricTensor):
                    cache[n] = output.tensor.detach().clone()
                    cache[n + '__gtype'] = copy.deepcopy(output.type)
                else:
                    cache[n] = output.detach().clone()
            return _hook
        hooks.append(module.register_forward_hook(_make_hook(name)))

    with torch.no_grad():
        x_dev  = x.unsqueeze(0).to(device) if x.dim() == 3 else x.to(device)
        out    = model(x_dev)
        logits = out[0] if isinstance(out, tuple) else out

    for h in hooks:
        h.remove()
    return logits.cpu(), cache

def intervention_pass(model, x_clean, cache_gx, target_hook_name, hook_specs, device):
    """Patch target hook with gx's activation; run x_clean's graph forward."""
    from escnn.nn import GeometricTensor
    assert target_hook_name in cache_gx, f"Hook '{target_hook_name}' not in cache."
    injected = [False]
    hooks    = []
    def _inject_hook(mod, inp, output):
        if injected[0]: return output
        injected[0] = True
        cached = cache_gx[target_hook_name]
        if cache_gx.get(target_hook_name + '__is_tuple', False):
            return tuple(t.to(device) if isinstance(t, torch.Tensor) else t for t in cached)
        elif target_hook_name + '__gtype' in cache_gx:
            return GeometricTensor(cached.to(device), cache_gx[target_hook_name + '__gtype'])
        else:
            return cached.to(device)
    hooks.append(hook_specs[target_hook_name].register_forward_hook(_inject_hook))
    with torch.no_grad():
        x_dev = x_clean.unsqueeze(0).to(device) if x_clean.dim() == 3 else x_clean.to(device)
        out = model(x_dev)
        logits = out[0] if isinstance(out, tuple) else out
    for h in hooks: h.remove()
    return logits.cpu()


MI_SUBSET = []
for orig_idx in subset_indices:
    img, label = test_dataset[orig_idx]
    MI_SUBSET.append({'img': img, 'label': label, 'orig_idx': orig_idx})
print(f"Sampled {len(MI_SUBSET)} images for MI.")
vanilla_model.eval()
for sample in MI_SUBSET:
    x = sample['img']
    for g_name in [g for g in GROUP_ELEMS if g != 'e']:
        xg = D4_TRANSFORMS[g_name](x.unsqueeze(0)).squeeze(0)
        _, cache = cache_activations(
            vanilla_model, xg, VANILLA_PROBE_HOOKS, DEVICE_PINN)
        for h in VANILLA_PROBE_HOOKS:
            act = cache.get(h)
            if act is None: continue
            if isinstance(act, tuple): act = act[0]
            feat = act.float().reshape(-1).cpu().numpy()
            _van_feats[h].append(feat)
        _van_labels.append(GROUP_ELEMS.index(g_name) - 1)


_van_labels = np.array(_van_labels)
print(f"Vanilla probe samples: {len(_van_labels)}")
print(f"Chance: {1/7:.4f}\n")

for h in VANILLA_PROBE_HOOKS:
    X = np.stack(_van_feats[h])
    if X.shape[1] > N_PCA:
        X = PCA(n_components=N_PCA).fit_transform(X)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    accs = [accuracy_score(
        _van_labels[te],
        LogisticRegression(C=0.1,max_iter=500).fit(
            X[tr], _van_labels[tr]).predict(X[te])
    ) for tr, te in cv.split(X, _van_labels)]
    print(f"Vanilla {h}: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    
print("\nCompare to D4 probe at same hooks")
print("If Vanilla H09 >> D4 H09: equivariance causally suppresses group info")

In [ ]:
vanilla_model.to(DEVICE_VANILLA)
vanilla_model.load_state_dict(torch.load(VANILLA_CKPT, map_location=DEVICE_VANILLA, weights_only=True))
vanilla_model.eval()
probs, labels = predict_no_tta(vanilla_model, test_loader, DEVICE_VANILLA, is_pinn=True)
auc_val = roc_auc_score(labels, probs, multi_class='ovr', average='macro')
print(f"VanillaLensPINN Macro AUC: {auc_val:.4f}")
assert auc_val >= 0.93, f"AUC {auc_val:.4f} below gate 0.93"


In [ ]:
# TTA evaluation for VanillaLensPINN
probs_tta, labels_tta = predict_with_tta(vanilla_model, test_loader, DEVICE_VANILLA, is_pinn=True)
auc_tta = roc_auc_score(labels_tta, probs_tta, multi_class='ovr', average='macro')
print(f"VanillaLensPINN TTA AUC: {auc_tta:.4f}")
print(f"TTA gain: {auc_tta - auc_val:+.4f}")
print("Note: if TTA hurts accuracy but not AUC, Sobel sign-flip is active in Vanilla too.")


In [ ]:
# Load or sample MI subset
labels_all = np.array([test_dataset[i][1] for i in range(len(test_dataset))])
subset_indices = []
counts = {0: 67, 1: 67, 2: 66}
torch.manual_seed(SEED); np.random.seed(SEED)
for cls, n in counts.items():
    cls_idx = np.where(labels_all == cls)[0]
    subset_indices.extend(np.random.choice(cls_idx, size=n, replace=False).tolist())
subset_indices = sorted(subset_indices)
MI_SUBSET = []
for orig_idx in subset_indices:
    img, label = test_dataset[orig_idx]
    MI_SUBSET.append({'img': img, 'label': label, 'orig_idx': orig_idx})
print(f"Sampled {len(MI_SUBSET)} images for MI.")


In [ ]:
VANILLA_HOOKS = {
    'H00_preprocess':    vanilla_model.preprocess,
    'H01_enc1':          vanilla_model.d4unet.enc1,
    'H02_enc2':          vanilla_model.d4unet.enc2,
    'H03_enc3':          vanilla_model.d4unet.enc3,
    'H04_bot':           vanilla_model.d4unet.bot,
    'H08_kappa_out':     vanilla_model.d4unet.kappa_out,
    'H09_poisson':       vanilla_model.poisson,
    'H10_deflection':    vanilla_model.deflection,
    'H11_inv_lens':      vanilla_model.inv_lens,
    'H12_HANDOFF':       vanilla_model.handoff_probe,
    'H13_eff3':          vanilla_model.classifier.features[3],
    'H13b_eff4':         vanilla_model.classifier.features[4],
    'H14_eff5':          vanilla_model.classifier.features[5],
    'H14b_eff6':         vanilla_model.classifier.features[6],
    'H15_before_gap':    vanilla_model.classifier.features[7],
    'H16_after_gap':     vanilla_model.classifier.avgpool,
    'H17_pre_linear':    vanilla_model.classifier.head[0],
}
VANILLA_HOOKS_PRIMARY = {k:v for k,v in VANILLA_HOOKS.items()}


In [ ]:
def cache_activations(model, x, hook_specs, device):
    """Single forward pass. Returns (logits, cache_dict)."""
    cache = {}
    hooks = []

    for name, module in hook_specs.items():
        def _make_hook(n):
            def _hook(mod, inp, output):
                if isinstance(output, tuple):
                    cache[n] = tuple(
                        t.detach().clone() if isinstance(t, torch.Tensor) else t
                        for t in output
                    )
                    cache[n + '__is_tuple'] = True
                elif isinstance(output, GeometricTensor):
                    cache[n] = output.tensor.detach().clone()
                    cache[n + '__gtype'] = copy.deepcopy(output.type)
                else:
                    cache[n] = output.detach().clone()
            return _hook
        hooks.append(module.register_forward_hook(_make_hook(name)))

    with torch.no_grad():
        x_dev  = x.unsqueeze(0).to(device) if x.dim() == 3 else x.to(device)
        out    = model(x_dev)
        logits = out[0] if isinstance(out, tuple) else out

    for h in hooks:
        h.remove()

    return logits.cpu(), cache


def intervention_pass(model, x_clean, cache_gx, target_hook_name, hook_specs, device):
    """Patch target hook with gx's activation; run x_clean's graph forward.
    inject_hook is a closure INSIDE this function — do NOT move to module level."""
    assert target_hook_name in cache_gx, (
        f"Hook '{target_hook_name}' not in gx_cache. "
        f"Available: {[k for k in cache_gx if not k.endswith('__gtype') and not k.endswith('__is_tuple')]}"
    )

    injected = [False]
    hooks    = []

    def _inject_hook(mod, inp, output):
        if injected[0]:
            return output
        injected[0] = True
        cached = cache_gx[target_hook_name]
        if cache_gx.get(target_hook_name + '__is_tuple', False):
            return tuple(t.to(device) if isinstance(t, torch.Tensor) else t for t in cached)
        elif target_hook_name + '__gtype' in cache_gx:
            gtype = cache_gx[target_hook_name + '__gtype']
            assert hasattr(gtype, 'representations'), (
                f"Cached FieldType at {target_hook_name} is corrupted between passes. "
                f"Re-run cache_activations for this image."
            )
            return GeometricTensor(cached.to(device), gtype)
        else:
            return cached.to(device)

    hooks.append(hook_specs[target_hook_name].register_forward_hook(_inject_hook))

    with torch.no_grad():
        x_dev  = x_clean.unsqueeze(0).to(device) if x_clean.dim() == 3 else x_clean.to(device)
        out    = model(x_dev)
        logits = out[0] if isinstance(out, tuple) else out

    for h in hooks:
        h.remove()

    return logits.cpu()


In [ ]:
# SANITY CHECK 5: kappa_hat NON-invariance (VanillaLensPINN control verification)
s0 = MI_SUBSET[0]

# Ensure transform lambda gets proper unsqueezed input then squished back to 3D.
img_x = s0['img'].unsqueeze(0).to(DEVICE_VANILLA)
img_gx = D4_TRANSFORMS['r90'](s0['img'].unsqueeze(0)).to(DEVICE_VANILLA)

# Preprocess into 2 channels (dx, dy) as expected by d4unet
xin = vanilla_model.preprocess(img_x)
xgin = vanilla_model.preprocess(img_gx)

kappa_x = vanilla_model.d4unet(xin).detach()
kappa_gx = vanilla_model.d4unet(xgin).detach()

# Manual alignment logic comparison
_g_kappa_x = torch.rot90(kappa_x, k=1, dims=[2, 3])
best_diff = (kappa_gx - _g_kappa_x).abs().mean().item()

if best_diff > 1e-3:
    print(f"  PASS: VanillaUNet kappa is NOT D4-invariant (best={best_diff:.5f}) — control confirmed")
else:
    print(f"  WARN: VanillaUNet kappa appears near-invariant (best={best_diff:.5f} < 1e-3). logic inversion? check SC5.")

assert best_diff > 1e-3, "VanillaUNet MUST be non-equivariant for this control"


In [ ]:
# SANITY CHECK 1: Identity patch → delta < 1e-3
s0 = MI_SUBSET[0]
logit_clean, cache_x = cache_activations(vanilla_model, s0['img'], VANILLA_HOOKS, DEVICE_VANILLA)
for h in list(VANILLA_HOOKS_PRIMARY.keys())[:5]:  # first 5 hooks sufficient
    lp = intervention_pass(vanilla_model, s0['img'], cache_x, h, VANILLA_HOOKS_PRIMARY, DEVICE_VANILLA)
    delta = (lp - logit_clean).norm(p=2).item()
    assert delta < 1e-3, f"Identity check FAILED at {h}: delta={delta:.4f}"
print("PASS SC1: Identity patch delta < 1e-3 for first 5 hooks")

# SANITY CHECK 3: CV > 0.1 (no graph contamination)
img_r90 = D4_TRANSFORMS['r90'](s0['img'].unsqueeze(0)).squeeze(0)
_, cache_r90 = cache_activations(vanilla_model, img_r90, VANILLA_HOOKS, DEVICE_VANILLA)
deltas_profile = []
for h in VANILLA_HOOKS_PRIMARY:
    lp = intervention_pass(vanilla_model, s0['img'], cache_r90, h, VANILLA_HOOKS_PRIMARY, DEVICE_VANILLA)
    deltas_profile.append((lp - logit_clean).norm(p=2).item())
import numpy as np
cv = np.std(deltas_profile) / (np.mean(deltas_profile) + 1e-8)
print(f"CV across hooks: {cv:.3f}")
assert cv > 0.1 or np.mean(deltas_profile) < 0.01, \
    "SC3 FAIL: Uniform deltas = graph contamination in intervention_pass"
print(f"PASS SC3: Non-uniform delta profile (CV={cv:.3f})")


In [ ]:
import datetime
import pandas as pd
import os

print("=" * 60)
print(f"VanillaLensPINN MI experiment started: {datetime.datetime.now().isoformat()}")
print("Pre-specified outcomes on record: A/B/C/D as per research writeup")
print("=" * 60)

results = []
for idx, s in enumerate(MI_SUBSET):
    if idx % 10 == 0: print(f"Processing {idx}/{len(MI_SUBSET)}")
    img = s['img']
    logit_clean, cache_clean = cache_activations(vanilla_model, img, VANILLA_HOOKS, DEVICE_VANILLA)
    for g_name in GROUP_ELEMS:
        
        # Ensure lambda transform is applied to the image BEFORE cache
        x_g = D4_TRANSFORMS[g_name](img.unsqueeze(0)).squeeze(0)
        _, cache_xg = cache_activations(vanilla_model, x_g, VANILLA_HOOKS, DEVICE_VANILLA)
        
        for h_name in VANILLA_HOOKS_PRIMARY.keys():
            logit_patched = intervention_pass(vanilla_model, img, cache_xg, h_name, VANILLA_HOOKS_PRIMARY, DEVICE_VANILLA)
            
            diff = (logit_patched - logit_clean).squeeze()
            delta_l2 = diff.norm(p=2).item()
            
            # act_diff_sq logic (optional inside loop internals)
            val_xg = cache_xg[h_name]
            val_cl = cache_clean[h_name]
            if isinstance(val_xg, tuple):
                act_diff = sum((a - b).pow(2).sum().item() for a, b in zip(val_xg, val_cl) if isinstance(a, torch.Tensor))
            else:
                act_diff = (val_xg - val_cl).pow(2).sum().item()

            results.append({
                'model': 'VanillaLensPINN',
                'img_idx': s['orig_idx'],        # user explicitly wanted img_idx
                'true_class': s['label'],       # user explicitly wanted true_class
                'group': g_name,                # user explicitly wanted group
                'hook': h_name,
                'delta_l2': delta_l2,           # user explicitly wanted delta_l2
                'act_diff_sq': act_diff         # kept for compatibility with analysis cell structure
            })

df_vanilla = pd.DataFrame(results)
out_dir = '/kaggle/working/mi_experiment_vanilla'
os.makedirs(out_dir, exist_ok=True)
out_csv = os.path.join(out_dir, 'mi_results_full.csv')             # User's analysis cell calls 'mi_results_vanilla.csv'

# Override full execution (as the user wants a clean standalone run output)
df_vanilla.to_csv(out_csv, mode='w', index=False, header=True)

print(f"MI Loop complete. Results saved to {out_csv}")


In [ ]:
import pandas as pd
import numpy as np
import os

_van_path = '/kaggle/working/mi_experiment_vanilla/mi_results_full.csv'
assert os.path.exists(_van_path), f"MISSING: {_van_path}"

df_van = pd.read_csv(_van_path)
df_van_prim = df_van[~df_van['secondary']] if 'secondary' in df_van.columns else df_van

# Identify GAP hooks — check what hook names exist
print("Available hooks in vanilla CSV:")
print(sorted(df_van_prim['hook'].unique()))

# Adapt these to match actual hook names in vanilla CSV
VANILLA_BEFORE_GAP = 'H15_before_gap'
VANILLA_AFTER_GAP  = 'H16_after_gap'

ratios = {}
for g in [x for x in df_van_prim['group'].unique() if x != 'e']:
    sub    = df_van_prim[df_van_prim['group'] == g]
    before = sub[sub['hook'] == VANILLA_BEFORE_GAP]['delta_l2'].mean()
    after  = sub[sub['hook'] == VANILLA_AFTER_GAP]['delta_l2'].mean()
    ratios[g] = (before - after) / before if before > 1e-6 else 1.0

votes_collapse = sum(1 for r in ratios.values() if r > 0.5)
votes_routing  = sum(1 for r in ratios.values() if r < 0.2)
verdict = ('COLLAPSE' if votes_collapse >= 4
           else 'ROUTING' if votes_routing >= 4
           else 'AMBIGUOUS')

print(f"\nVanillaLensPINN COMPUTED verdict: {verdict}")
print(f"  collapse votes: {votes_collapse}/7")
print(f"  routing  votes: {votes_routing}/7")
print(f"  mean GAP ratio: {np.mean(list(ratios.values())):.4f}")
print(f"\nD4LensPINN physics pipeline comparison:")
for h in ['H09_poisson', 'H11_inv_lens', 'H12_HANDOFF']:
    van_val = df_van_prim[
        (df_van_prim['hook']==h) & 
        (df_van_prim['group']!='e')
    ]['delta_l2'].mean()
    print(f"  {h}: Vanilla={van_val:.4f}")
print("  Compare to D4: H09=0.35, H11=15.75, H12=2.78")


In [ ]:
# Add to bootstrap cell:
_df_van = pd.read_csv('/kaggle/working/mi_experiment_vanilla/mi_results_full.csv')
_df_d4  = pd.read_csv(os.path.join('/kaggle/input/datasets/[NAME]26189/d4-csv/mi_results_full (1).csv'))
_rng    = np.random.default_rng(SEED)

def _bci(vals, n=2000):
    return np.percentile(
        [_rng.choice(vals,size=len(vals),replace=True).mean() 
         for _ in range(n)], [2.5,97.5])

print("Bootstrap 95% CIs — Physics Pipeline Differential\n")
for _h, _label in [
    ('H09_poisson',  'Poisson'),
    ('H11_inv_lens', 'InverseLens'),
    ('H12_HANDOFF',  'HANDOFF'),
]:
    for _model_name, _df in [('D4', _df_d4), ('Vanilla', _df_van)]:
        _vals = _df[
            (_df['hook']==_h) & (_df['group']!='e')
        ]['delta_l2'].values
        if len(_vals) == 0:
            print(f"  {_model_name} {_label}: NO DATA")
            continue
        _lo, _hi = _bci(_vals)
        print(f"  {_model_name:8s} {_label:15s}: "
              f"{_vals.mean():.4f} [95% CI {_lo:.4f}, {_hi:.4f}]")
    print()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

# ✅ ADDED: Define and load the actual results from disk
_csv = '/kaggle/working/mi_experiment_vanilla/mi_results_full.csv'
if not os.path.exists(_csv):
    print(f"ERROR: Results file not found at {_csv}. Please run the MI Loop cell first.")
else:
    df = pd.read_csv(_csv)
    
    # Now your original logic works!
    df_ne = df[df['group'] != 'e'].copy()

    HOOK_ORDER = [
        'H00_preprocess','H01_enc1','H02_enc2','H03_enc3','H04_bot',
        'H08_kappa_out','H09_poisson','H10_deflection','H11_inv_lens',
        'H12_HANDOFF','H12b_input_proj','H13_eff3','H13b_eff4',
        'H14_eff5','H14b_eff6','H15_before_gap','H16_after_gap','H17_pre_linear'
    ]
    # Keep only hooks that actually exist in the CSV
    HOOK_ORDER = [h for h in HOOK_ORDER if h in df_ne['hook'].unique()]

    COLORS = {
        'r90':'#a8c8f0','r180':'#4a90d9','r270':'#1a4fa8',
        'flip_h':'#f0a8a8','flip_h_r90':'#d45a5a',
        'flip_h_r180':'#a81a1a','flip_h_r270':'#5c0000',
    }
    LINESTYLES = {g: '-' if not g.startswith('flip') else '--' for g in COLORS}

    agg = (df_ne.groupby(['group','hook'])['delta_l2']
           .agg(['mean','std']).reset_index())

    fig, ax = plt.subplots(figsize=(16, 5))
    for g, color in COLORS.items():
        if g not in agg['group'].unique(): continue  # Guard for missing groups
        g_data = agg[agg['group'] == g].set_index('hook')
        means = [g_data.loc[h,'mean'] if h in g_data.index else np.nan for h in HOOK_ORDER]
        stds  = [g_data.loc[h,'std']  if h in g_data.index else np.nan for h in HOOK_ORDER]
        ax.errorbar(range(len(HOOK_ORDER)), means, yerr=stds,
                    label=g, color=color, linestyle=LINESTYLES[g],
                    linewidth=1.5, marker='o', markersize=4, capsize=3, alpha=0.85)

    if 'H11_inv_lens' in HOOK_ORDER:
        ax.axvline(HOOK_ORDER.index('H11_inv_lens'), color='darkorange', linestyle='--', label='InverseLensLayer')
    if 'H16_after_gap' in HOOK_ORDER:
        ax.axvline(HOOK_ORDER.index('H16_after_gap'), color='darkgreen', linestyle='--', label='GAP')

    ax.set_xticks(range(len(HOOK_ORDER)))
    ax.set_xticklabels(HOOK_ORDER, rotation=45, ha='right', fontsize=7)
    ax.set_ylabel('Mean Logit Delta (L2)')
    ax.set_title('VanillaLensPINN — Mean Logit Delta Under D4 Interventions\n(N=200 images, ±1 std)', fontsize=11, fontweight='bold')
    ax.legend(fontsize=7, ncol=4)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('vanilla_figure1.pdf', dpi=300)
    plt.savefig('vanilla_figure1.png', dpi=300)
    plt.show()
    print("Figure saved.")


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

N_PCA_TARGET = 100
VANILLA_PROBE_HOOKS = {
    'H09_poisson': vanilla_model.poisson,
    'H12_handoff': vanilla_model.handoff_probe,
}

_van_feats = {h: [] for h in VANILLA_PROBE_HOOKS}
_van_labels = []
GROUP_NO_E = [g for g in GROUP_ELEMS if g != 'e']

vanilla_model.eval()
for sample in MI_SUBSET:
    x = sample['img']
    for g_name in GROUP_NO_E:
        xg = D4_TRANSFORMS[g_name](x.unsqueeze(0)).squeeze(0)
        _, cache = cache_activations(vanilla_model, xg, VANILLA_PROBE_HOOKS, DEVICE_VANILLA)
        for h in VANILLA_PROBE_HOOKS:
            act = cache.get(h)
            if act is None:
                continue
            if isinstance(act, tuple):
                act = act[0]
            act = act.float()
            # ✅ KEY FIX: pool spatial dims to prevent 90,000-dim features
            if act.dim() == 4:
                act = act.mean(dim=[-2, -1])  # (B,C,H,W) → (B,C)
            feat = act.reshape(-1).cpu().numpy()
            _van_feats[h].append(feat)
        _van_labels.append(GROUP_NO_E.index(g_name))

_van_labels = np.array(_van_labels, dtype=np.int64)
n_classes = len(GROUP_NO_E)
chance = 1.0 / n_classes
print(f"Vanilla probe samples: {len(_van_labels)}, chance: {chance:.4f}")

# Print feature dims so you can verify
for h in VANILLA_PROBE_HOOKS:
    print(f"[{h}] feature dim after pooling: {np.stack(_van_feats[h]).shape}")

fold_rows, summary_rows = [], []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

for h in VANILLA_PROBE_HOOKS:
    X = np.stack(_van_feats[h])
    y = _van_labels.copy()

    n_comp = min(N_PCA_TARGET, X.shape[1], max(1, X.shape[0] - 1))
    if X.shape[1] > N_PCA_TARGET and n_comp >= 2:
        # ✅ randomized solver — fast even if pooling still leaves >100 dims
        X_use = PCA(n_components=n_comp, random_state=SEED,
                    svd_solver='randomized').fit_transform(X)
    else:
        X_use = X

    accs = []
    for fold_idx, (tr, te) in enumerate(cv.split(X_use, y), start=1):
        clf = LogisticRegression(C=0.1, max_iter=500, random_state=SEED)
        clf.fit(X_use[tr], y[tr])
        acc = accuracy_score(y[te], clf.predict(X_use[te]))
        accs.append(acc)
        fold_rows.append({
            'model': 'VanillaLensPINN', 'hook': h, 'fold': fold_idx,
            'accuracy': float(acc), 'n_train': int(len(tr)), 'n_test': int(len(te)),
        })

    mean_acc, std_acc = float(np.mean(accs)), float(np.std(accs))
    summary_rows.append({
        'model': 'VanillaLensPINN', 'hook': h,
        'mean_accuracy': mean_acc, 'std_accuracy': std_acc,
        'chance': float(chance), 'delta_over_chance': float(mean_acc - chance),
        'n_samples': int(X.shape[0]), 'raw_features': int(X.shape[1]),
        'pca_components_used': int(X_use.shape[1]), 'n_splits': 5,
    })
    print(f"Vanilla {h}: {mean_acc:.4f} +/- {std_acc:.4f}")

df_probe_fold = pd.DataFrame(fold_rows)
df_probe_summary = pd.DataFrame(summary_rows)

out_dir = '/kaggle/working/mi_experiment_vanilla'
os.makedirs(out_dir, exist_ok=True)
df_probe_fold.to_csv(os.path.join(out_dir, 'vanilla_probe_cv_per_fold.csv'), index=False)
df_probe_summary.to_csv(os.path.join(out_dir, 'vanilla_probe_summary.csv'), index=False)
print("Done.")
df_probe_summary

In [ ]:
import os
import zipfile

ROOT = '/kaggle/working'
ZIP_NAME = 'vanilla_mi_results.zip'
zip_path = os.path.join(ROOT, ZIP_NAME)

EXCLUDE_DIRS  = {'dataset', '__MACOSX', '.virtual_documents'}
EXCLUDE_FILES = {'dataset.zip', '.extracted', ZIP_NAME}

# FIX 1: remove stale zip before writing to avoid self-inclusion
if os.path.exists(zip_path):
    os.remove(zip_path)

added = 0
skipped = 0

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for cur_root, dirs, files in os.walk(ROOT):
        dirs[:] = [d for d in dirs if d not in EXCLUDE_DIRS]  # prune traversal

        for fname in files:
            if fname in EXCLUDE_FILES:
                skipped += 1
                continue

            abs_path = os.path.join(cur_root, fname)
            rel_path = os.path.relpath(abs_path, ROOT)

            # FIX 3: guard against permission errors on Kaggle system files
            try:
                zf.write(abs_path, arcname=rel_path)
                added += 1
            except (PermissionError, OSError) as e:
                print(f"Skipped (unreadable): {rel_path} — {e}")
                skipped += 1

print(f"MI results zipped: {zip_path}")
print(f"Added files: {added} | Skipped excluded files: {skipped}")